# 03d — Weather Enrichment (Problem C)

Enrich the case-control dataset with historical weather data (METAR) using the `Meteostat` library.

**Goal:** For every departure in our dataset, fetch the closest hourly weather observation.
**Output:** `data/processed/preflight_weather_enriched.parquet`

In [4]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from meteostat import Hourly, Stations
from tqdm.auto import tqdm
from pathlib import Path

IN_PATH = Path('data/processed/preflight_casecontrol.parquet')
OUT_PATH = Path('data/processed/preflight_weather_enriched.parquet')

df = pd.read_parquet(IN_PATH)
print(f"Loaded dataset with {len(df):,} rows.")

Loaded dataset with 741,394 rows.


## 1. Prepare Timestamps and Station Mapping

We need to convert the BTS date and time into a proper UTC or Local timestamp for querying Meteostat.

In [6]:
def parse_bts_time(date_str, time_float):
    if pd.isna(time_float):
        return pd.NaT
    # BTS time is HHMM as float (e.g. 1015.0 for 10:15)
    t_str = f"{int(time_float):04d}"
    hours = int(t_str[:2]) % 24
    mins = int(t_str[2:])
    return pd.to_datetime(date_str) + timedelta(hours=hours, minutes=mins)

df['departure_timestamp'] = df.apply(lambda row: parse_bts_time(row['FL_DATE'], row['DEP_TIME']), axis=1)
print(f"Parsed {df['departure_timestamp'].notna().sum():,} timestamps.")

Parsed 727,650 timestamps.


## 2. Weather Fetching Logic

To avoid overwhelming the API, we cluster departures by **Airport + Date** and fetch the full hourly weather for that day/airport once.

In [9]:
def get_weather_for_batch(airport_code, start_date, end_date):
    """Fetch hourly weather for an airport station over a date range."""
    # Find the closest station for this airport code (IATA)
    stations = Stations()
    stations = stations.inventory('hourly', (start_date, end_date))
    station = stations.nearby_iata(airport_code).fetch(1)
    
    if station.empty:
        return pd.DataFrame()
    
    data = Hourly(station.index[0], start_date, end_date)
    return data.fetch()

In [8]:
# This step is slow on first run because it caches data locally.
# We'll group by airport and fetch in chunks.
unique_airports = df['ORIGIN'].unique()
print(f"Fetching weather for {len(unique_airports)} airports...")

weather_data = []
start_all = df['departure_timestamp'].min()
end_all = df['departure_timestamp'].max()

# For a POC, we might just fetch the full range for the top airports
# But for accuracy, we iterate. 
airport_weather_cache = {}

for airport in tqdm(unique_airports):
    try:
        w = get_weather_for_batch(airport, start_all, end_all)
        if not w.empty:
            w['ORIGIN'] = airport
            weather_data.append(w)
    except Exception as e:
        continue

full_weather_df = pd.concat(weather_data).reset_index()
full_weather_df = full_weather_df.rename(columns={'time': 'weather_time'})
print(f"Downloaded {len(full_weather_df):,} hourly weather observations.")

Fetching weather for 370 airports...


  0%|          | 0/370 [00:00<?, ?it/s]

ValueError: No objects to concatenate

## 3. Merge Weather with Dataset

We merge by matching the flight departure time to the **nearest hour** in the weather data.

In [10]:
# Round departure time to nearest hour
df['weather_lookup_time'] = df['departure_timestamp'].dt.round('H')

enriched_df = pd.merge(
    df,
    full_weather_df,
    left_on=['ORIGIN', 'weather_lookup_time'],
    right_on=['ORIGIN', 'weather_time'],
    how='left'
)

print(f"Enrichment complete. Row count: {len(enriched_df)}")
print(f"Weather coverage: {enriched_df['temp'].notna().mean()*100:.1f}%")

NameError: name 'full_weather_df' is not defined

## 4. Handle Missing Data and Inspect

In [ ]:
# Fill missing weather with daily averages or forward fill if needed
# For now, we'll just note the missingness.
missing_mask = enriched_df['temp'].isna()
print(f"Rows missing weather: {missing_mask.sum()}")

# Basic stats check
display(enriched_df[['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']].describe())

## 5. Save Enriched Dataset

In [ ]:
enriched_df.to_parquet(OUT_PATH, index=False)
print(f"Saved weather-enriched dataset to {OUT_PATH}")